In [ ]:
import numpy as np
from scipy.optimize import linprog
from scipy.linalg import solve
import cvxpy as cp
from scipy.optimize import minimize, linprog
from itertools import combinations, permutations
import time, warnings

#**Conjecture**

In [ ]:
warnings.filterwarnings("ignore")

# ----------  exact StQP  ----------
def p_star_exact(Q, tol=1e-10):
    n = 5
    best = min(Q[i,i] for i in range(n))
    for r in range(2, n+1):
        for S in combinations(range(n), r):
            S = list(S)
            k = len(S)
            A = np.zeros((k+1, k+1))
            b = np.zeros(k+1)
            for idx, i in enumerate(S):
                for jdx, j in enumerate(S):
                    A[idx, jdx] = Q[i, j]
                A[idx, k] = -1.0
            A[k, :k] = 1.0
            b[k] = 1.0
            try:
                sol = np.linalg.solve(A, b)
                x_vec = sol[:k]
                lam = sol[k]
                if np.all(x_vec >= -tol):
                    x = np.zeros(n)
                    for idx, i in enumerate(S):
                        x[i] = x_vec[idx]
                    Qx = Q @ x
                    if np.all(Qx[~np.isin(np.arange(n), S)] >= lam - tol):
                        best = min(best, lam)
                        continue
            except np.linalg.LinAlgError:
                pass
            c = np.zeros(k+1); c[-1] = 1.0
            bounds_lp = [(0, None)] * k + [(None, None)]
            res = linprog(c, A_eq=A, b_eq=b, bounds=bounds_lp,
                          method='highs', options={'disp': False})
            if res.success:
                lam = res.x[-1]
                x = np.zeros(n)
                for idx, i in enumerate(S):
                    x[i] = res.x[idx]
                Qx = Q @ x
                if np.all(Qx[~np.isin(np.arange(n), S)] >= lam - tol):
                    best = min(best, lam)
    return best

# ----------  DNN primal  ----------
def p_dnn(Q, eps=1e-9, max_iters=30000):
    n = 5
    X = cp.Variable((n,n), symmetric=True)
    constraints = [X >> 0, X >= 0, cp.sum(X) == 1]
    obj = cp.Minimize(cp.sum(cp.multiply(Q, X)))
    prob = cp.Problem(obj, constraints)
    prob.solve(solver=cp.SCS, eps=eps, max_iters=max_iters, verbose=False)
    return prob.value

# ----------  matrix assembly  ----------
def vec_to_Q(params):
    n = 5
    Q = np.zeros((n,n))
    for i in range(n):
        Q[i,i] = params[i]
    for i in range(n):
        e = params[5 + i]
        Q[i, (i+1)%n] = e
        Q[(i+1)%n, i] = e
    for i in range(n):
        f = params[10 + i]
        Q[i, (i+2)%n] = f
        Q[(i+2)%n, i] = f
    return Q

# ----------  search objective  ----------
def neg_ratio_search(params):
    t0 = time.time()
    if np.any(params < 0):
        return 1e6
    Q = vec_to_Q(params)
    t1 = time.time()
    ps = p_star_exact(Q)
    t2 = time.time()
    pdnn = p_dnn(Q, eps=1e-6, max_iters=10000)
    t3 = time.time()
    if pdnn is None or pdnn < 1e-8:
        return 1e6
    ratio = ps / pdnn
    t4 = time.time()
    # print per‑call timing
    #print(f"vec {t1-t0:.4f}, StQP {t2-t1:.4f}, DNN {t3-t2:.4f}, total {t4-t0:.4f}")
    return -ratio

# ----------  final objective  ----------
def neg_ratio_accurate(params):
    if np.any(params < 0):
        return 1e6
    Q = vec_to_Q(params)
    ps = p_star_exact(Q)
    pdnn = p_dnn(Q, eps=1e-9, max_iters=50000)
    if pdnn is None or pdnn < 1e-8:
        return 1e6
    return - (ps / pdnn)

# ----------  circulant check  ----------
def is_circulant_perm(Q, tol=1e-6):
    n = Q.shape[0]
    for perm in permutations(range(n)):
        P = np.zeros((n,n))
        for i,j in enumerate(perm):
            P[i,j] = 1
        Qp = P @ Q @ P.T
        d = Qp[0,0]; e = Qp[0,1]; f = Qp[0,2]
        if abs(Qp[0,4]-e) > tol: continue
        if abs(Qp[0,3]-f) > tol: continue
        ok = True
        for i in range(n):
            if abs(Qp[i,i]-d) > tol or \
               abs(Qp[i,(i+1)%n]-e) > tol or abs(Qp[(i+1)%n,i]-e) > tol or \
               abs(Qp[i,(i+2)%n]-f) > tol or abs(Qp[(i+2)%n,i]-f) > tol:
                ok = False; break
        if ok:
            return True, perm, (d,e,f)
    return False, None, None

# ----------  dual lower bound  ----------
def dual_lower_bound(Q, eps=1e-10, max_iters=50000):
    n = Q.shape[0]
    lam = cp.Variable()
    S = cp.Variable((n,n), symmetric=True)
    N = cp.Variable((n,n))
    constraints = [Q - lam * np.ones((n,n)) == S + N, S >> 0, N >= 0]
    prob = cp.Problem(cp.Maximize(lam), constraints)
    prob.solve(solver=cp.SCS, eps=eps, max_iters=max_iters, verbose=False)
    if prob.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
        return lam.value
    return None

# ========================  SMART MULTISTART  ====================
bound = np.sqrt(5)/2
dim = 15
np.random.seed(42)
num_restarts = 100
maxiter = 200
best_ratio = 0.0
best_params = None

sqrt5 = np.sqrt(5)
start_time = time.time()

for r in range(num_restarts):
    # Half near circulant extremal, half random
    if r < num_restarts // 2:
        scale = 1.0 + 0.5 * np.random.randn()
        scale = max(0.5, min(3.0, scale))
        x0 = np.zeros(dim)
        x0[0:5] = sqrt5 * scale
        x0[5:10] = sqrt5 * scale
        x0[10:] = 0.0
        x0 += 0.1 * np.random.randn(dim)
        x0 = np.maximum(x0, 0.0)
    else:
        x0 = np.random.rand(dim) * 5

    res = minimize(neg_ratio_search, x0, method='Nelder-Mead',
                   options={'maxiter': maxiter, 'disp': False})
    current = -res.fun
    if current > best_ratio:
        best_ratio = current
        best_params = res.x.copy()
        print(f"Restart {r+1:2d}: new best = {best_ratio:.10f}")

    if (r+1)%10==0:
      print('Iter:', r+1)
search_time = time.time() - start_time
print(f"\nSearch finished in {search_time:.1f}s")
print(f"Best ratio (search accuracy) = {best_ratio:.10f}")

# ---- High‑accuracy final evaluation ----
print("\nEvaluating best candidate with high accuracy ...")
best_ratio_accurate = -neg_ratio_accurate(best_params)
print(f"Best ratio (high accuracy) = {best_ratio_accurate:.10f}")
print(f"Conjectured bound (√5/2)   = {bound:.10f}")

# ---- Dual verification if suspicious ----
if best_ratio_accurate > bound + 1e-7:
    print("\nRatio above bound. Running dual verification...")
    Q_best = vec_to_Q(best_params)
    lam_dual = dual_lower_bound(Q_best)
    if lam_dual is not None:
        ratio_dual = p_star_exact(Q_best) / lam_dual
        print(f"Certified ratio (via dual) = {ratio_dual:.10f}")
        if ratio_dual > bound + 1e-8:
            print("  => CONFIRMED COUNTEREXAMPLE. Conjecture is FALSE.")
        else:
            print("  => Dual bound shows ratio within tolerance; SDP noise.")
    else:
        print("  Dual solver failed.")
else:
    print("Ratio does not exceed bound. Conjecture strongly supported.")

# ---- Circulant check ----
Q_best = vec_to_Q(best_params)
is_circ, perm, vals = is_circulant_perm(Q_best)
print(f"\nBest matrix circulant? {is_circ}")
if is_circ:
    print(f"  permutation {perm}, d,e,f = {vals}")
print("\nBest matrix Q:")
print(Q_best)

#**Circulant Matrices**

In [ ]:
# ---------- Exact StQP (KKT enumeration) ----------
def p_star_exact(Q, tol=1e-10):
    n = Q.shape[0]
    best = np.inf
    for r in range(1, n+1):
        for S in combinations(range(n), r):
            S = list(S)
            k = len(S)
            A = np.zeros((k+1, k+1))
            b = np.zeros(k+1)
            for idx, i in enumerate(S):
                for jdx, j in enumerate(S):
                    A[idx, jdx] = Q[i, j]
                A[idx, k] = -1.0
            A[k, :k] = 1.0
            b[k] = 1.0

            # Try direct solve first
            try:
                sol = np.linalg.solve(A, b)
                x_vec = sol[:k]
                lam = sol[k]
                if np.all(x_vec >= -tol):
                    x_full = np.zeros(n)
                    for idx, i in enumerate(S):
                        x_full[i] = x_vec[idx]
                    Qx = Q @ x_full
                    if np.all(Qx[~np.isin(np.arange(n), S)] >= lam - tol):
                        best = min(best, lam)
                        continue
            except np.linalg.LinAlgError:
                pass   # singular, fall back to LP

            # Fallback: solve LP for this S
            # Variables: x (size k), λ (scalar)
            c = np.zeros(k+1)
            c[-1] = 1.0
            A_eq = A
            b_eq = b
            bounds = [(0, None)] * k + [(None, None)]
            res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds,
                          method='highs', options={'disp': False})
            if res.success:
                lam = res.x[-1]
                x_full = np.zeros(n)
                for idx, i in enumerate(S):
                    x_full[i] = res.x[idx]
                Qx = Q @ x_full
                if np.all(Qx[~np.isin(np.arange(n), S)] >= lam - tol):
                    best = min(best, lam)
    return best

# ---------- Exact p_DNN for circulant 5x5 -------------
def p_dnn_circulant(d, e, f):
    sqrt5 = np.sqrt(5)
    L1 = d
    L2 = sqrt5/5 * d + (1 - sqrt5/5) * f
    L3 = (d + 2*e + 2*f) / 5
    L4 = sqrt5/5 * d + (1 - sqrt5/5) * e
    return min(L1, L2, L3, L4)

# ---------- Build circulant matrix ----------
def circulant_matrix(d, e, f):
    # Q[i,i]=d, Q[i,i±1]=e, Q[i,i±2]=f (indices mod 5)
    Q = np.zeros((5,5))
    for i in range(5):
        Q[i,i] = d
        Q[i, (i+1)%5] = e
        Q[(i+1)%5, i] = e
        Q[i, (i+2)%5] = f
        Q[(i+2)%5, i] = f
    return Q

# ---------- Test the extremal matrix ----------
sqrt5 = np.sqrt(5)
d_ext = sqrt5
e_ext = sqrt5
f_ext = 0.0
Q_ext = circulant_matrix(d_ext, e_ext, f_ext)

ps_ext = p_star_exact(Q_ext)
pdnn_ext = p_dnn_circulant(d_ext, e_ext, f_ext)
ratio_ext = ps_ext / pdnn_ext if pdnn_ext > 1e-12 else float('inf')

print("Extremal circulant matrix (d=√5, e=√5, f=0):")
print(f"  p* = {ps_ext:.10f}")
print(f"  p_DNN = {pdnn_ext:.10f}")
print(f"  Ratio = {ratio_ext:.10f}")
print(f"  Expected = {sqrt5/2:.10f}\n")

# ---------- Random circulant matrices ----------
np.random.seed(123)
max_ratio = 0.0
best_triple = None
trials = 101000

for i in range(trials):
    # generate random nonnegative d,e,f
    d = np.random.rand() * 5      # scale arbitrary
    e = np.random.rand() * 5
    f = np.random.rand() * 5
    Q = circulant_matrix(d, e, f)
    pdnn = p_dnn_circulant(d, e, f)
    if pdnn < 1e-12:
        continue
    ps = p_star_exact(Q)
    ratio = ps / pdnn

    if ratio > max_ratio:
        max_ratio = ratio
        #print("max ratio:", max_ratio)
        best_triple = (d, e, f)

    if (i+1) % 100 == 0:
        print(f"Iter {i+1}: current max ratio = {max_ratio:.8f}")

print(f"After {trials} random circulant matrices:")
print(f"  Max ratio = {max_ratio:.10f}")
print(f"  Achieved at d={best_triple[0]:.4f}, e={best_triple[1]:.4f}, f={best_triple[2]:.4f}")
print(f"  Conjectured bound = {sqrt5/2:.10f}")